# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [15]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [14]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Add uv to PATH for this notebook session
# import os
# os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]

# # Check uv works
# !uv --version

# # Create venv
# !uv venv .venv --seed

# # Install dependencies into the venv
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart kernel, then select: Python (cse151b)")

### Run the cell below every time to activate the installed environment. 

In [1]:
# # activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

In [1]:
# import sys
# print(sys.executable)

/workspace/151B_SP26_Competition/.venv/bin/python


In [13]:
# import sys, subprocess

# pkgs = ["torch", "transformers", "vllm", "bitsandbytes"]

# for pkg in pkgs:
#     print(f"\nTesting {pkg}...")
#     result = subprocess.run(
#         [sys.executable, "-c", f"import {pkg}; print('{pkg} OK')"],
#         capture_output=True,
#         text=True,
#         timeout=60,
#     )
#     print(result.stdout)
#     print(result.stderr)

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [1]:
print("Task started...")

import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

print("Task complete!")

Task started...
Task complete!


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [2]:
print("Task started...")

data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

print("Task complete!")

Task started...
Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}
Task complete!


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [3]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [16]:
# import sys

# !{sys.executable} -m pip uninstall -y vllm
# !{sys.executable} -m pip install "vllm==0.19.1"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 05-01 08:15:32 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.5, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-01 08:15:33 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 05-01 08:15:33 [model.py:1678] Using max model len 16384
WARNING 05-01 08:15:33 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=14402) INFO 05-01 08:17:18 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokeniz

(EngineCore pid=14402) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=14402) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(EngineCore pid=14402) INFO 05-01 08:17:36 [bitsandbytes_loader.py:786] Loading weights with BitsAndBytes quantization. May take a while ...


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.34it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.43it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.04it/s]
(EngineCore pid=14402) 


(EngineCore pid=14402) INFO 05-01 08:17:38 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 13.601413 seconds
(EngineCore pid=14402) INFO 05-01 08:17:58 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/8c9f128ca0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=14402) INFO 05-01 08:17:58 [backends.py:1111] Dynamo bytecode transform time: 19.92 s
(EngineCore pid=14402) INFO 05-01 08:18:03 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 32768) from the cache, took 4.173 s
(EngineCore pid=14402) INFO 05-01 08:18:03 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/8c300eb6da3e0ead728b359a7a6320e1b3efb48edaff76c86977facba6772916/rank_0_0/model
(EngineCore pid=14402) INFO 05-01 08:18:03 [monitor.py:48] torch.compile took 24.42 s in total
(EngineCore pid=14402) INFO 05-01 08:18:03 [monitor.py:76] Initial profiling/warmup run took 0.18 s
(Engin

(EngineCore pid=14402) 2026-05-01 08:18:06,185 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=14402) 2026-05-01 08:18:06,275 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 12.46it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 16.94it/s]


(EngineCore pid=14402) INFO 05-01 08:18:13 [gpu_model_runner.py:6046] Graph capturing finished in 7 secs, took 0.70 GiB
(EngineCore pid=14402) INFO 05-01 08:18:13 [gpu_worker.py:597] CUDA graph pool memory: 0.7 GiB (actual), 0.49 GiB (estimated), difference: 0.21 GiB (29.3%).
(EngineCore pid=14402) INFO 05-01 08:18:13 [core.py:283] init engine (profile, create kv cache, warmup model) took 34.54 seconds


(EngineCore pid=14402) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=14402) INFO 05-01 08:18:14 [vllm.py:790] Asynchronous scheduling is enabled.
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [17]:
# import sys
# !{sys.executable} -m pip install -U accelerate

In [6]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [9]:
# Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 5 questions...


Rendering prompts:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so I need to find the sum of the first 325 positive even whole numbers. Hmm, let's start by recalling what the first few even positive whole numbers are. The first one is 2, right? Then 4, 6, 8, and so on. Wait, positive even whole numbers, so they start at 2 and g ...

── Response 1 (id=1) ──
Okay, let's try to figure out this integral. The problem is the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s^2 + a^2) ds. Hmm, first, I need to recall how to integrate functions like this. 

First, let's note that a is probably a positive constant here because we have a^(3/2), so a should be positive to avoid complex numbers. The integral is over the entire real  ...

── Response 2 (id=2) ──
Okay, let's try to solve this problem step by step. First, part (a) is about the turkey cooling down, so I think th

### Generate with Transformers (for Datahub)

In [18]:
# from transformers import TextStreamer

# import torch
# print(torch.cuda.is_available(), "\n")
# print(torch.cuda.get_device_name(0), "\n")
# print(next(llm.parameters()).device, "\n")

# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [
#             {"role": "system", "content": system},
#             {"role": "user", "content": user},
#         ],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# responses = []

# for i, prompt in enumerate(prompts):
#     print(f"\n── Generating Response {i} (id={data[i].get('id')}) ──")

#     inputs = tokenizer(
#         prompt,
#         return_tensors="pt",
#         truncation=True,
#         max_length=16384,
#     ).to(llm.device)

#     streamer = TextStreamer(
#         tokenizer,
#         skip_prompt=True,
#         skip_special_tokens=True,
#     )

#     with torch.no_grad():
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             repetition_penalty=1.0,
#             do_sample=True,
#             streamer=streamer,
#         )

#     # Decode only new tokens
#     new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
#     response = tokenizer.decode(
#         new_tokens,
#         skip_special_tokens=True,
#     ).strip()

#     responses.append(response)

#     print(f"\n── Finished Response {i} ──")

In [19]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.inference_mode():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         do_sample=True,
#         repetition_penalty=1.0,
#         pad_token_id=tokenizer.eos_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [20]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 5/1126 [00:00<01:14, 14.96it/s]

Scoring complete. 5 results.


## 8. Summary

Print accuracy broken down by question type.

In [21]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    2  (50.00%)
  Free-form  :    2 /    3  (66.67%)
  Overall    :    3 /    5  (60.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [22]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 5 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [1]:
import sys
print(sys.executable)
# Expect "/workspace/151B_SP26_Competition/.venv/bin/python"

/workspace/151B_SP26_Competition/.venv/bin/python


In [2]:
print("Cell 1: imports, config, data, prompts...")

import os
import re
import sys
import gc
import json
import time
import shutil
import subprocess
from pathlib import Path
from typing import Optional

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

# ── Main run config ─────────────────────────────────────────────
RUN_NAME = "vllm_run2"   # CHANGE THIS to resume/save from a different file

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = "data/public.jsonl"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESPONSE_PATH = RESULTS_DIR / f"{RUN_NAME}_responses.jsonl"
ATTEMPT_PATH = RESULTS_DIR / f"{RUN_NAME}_attempts.jsonl"
EVAL_PATH = RESULTS_DIR / f"{RUN_NAME}_eval.jsonl"
SUMMARY_PATH = RESULTS_DIR / f"{RUN_NAME}_summary.json"

# ── GPU config ─────────────────────────────────────────────────
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

# ── vLLM/model config ──────────────────────────────────────────
MAX_INPUT_LEN = 8192
MAX_MODEL_LEN = 8192

GPU_MEMORY_UTILIZATION = 0.50
MAX_NUM_SEQS = 256
MAX_NUM_BATCHED_TOKENS = 32768

# ── Generation config ──────────────────────────────────────────
# Qwen3-4B-Thinking-2507 is thinking-only, so it needs enough output tokens
# to think, emit </think>, then emit final answer.
MAX_TOKENS_MCQ = 16384
MAX_TOKENS_FRQ = 16384

BATCH_SIZE_FRQ = 2
BATCH_SIZE_MCQ = 4

# Whether to save outputs that never reach final answer format.
# Keep False so bad/incomplete generations can be retried later with more tokens.
SAVE_INCOMPLETE = False

print(f"RUN_NAME                 = {RUN_NAME}")
print(f"RESPONSE_PATH            = {RESPONSE_PATH}")
print(f"ATTEMPT_PATH             = {ATTEMPT_PATH}")
print(f"EVAL_PATH                = {EVAL_PATH}")
print(f"MAX_INPUT_LEN            = {MAX_INPUT_LEN}")
print(f"MAX_MODEL_LEN            = {MAX_MODEL_LEN}")
print(f"MCQ batch/toks           = {BATCH_SIZE_MCQ}, {MAX_TOKENS_MCQ}")
print(f"FRQ batch/toks           = {BATCH_SIZE_FRQ}, {MAX_TOKENS_FRQ}")
print(f"GPU_MEMORY_UTILIZATION   = {GPU_MEMORY_UTILIZATION}")

# ── Load data ──────────────────────────────────────────────────
t0 = time.time()

with open(DATA_PATH, "r") as f:
    data = [json.loads(line) for line in f]

# Limit number of questions for testing
TEST_LIMIT = 16   # set to None for full dataset

if TEST_LIMIT is not None:
    data = data[:TEST_LIMIT]

n_mcq = sum(bool(d.get("options")) for d in data)
n_frq = len(data) - n_mcq

print(f"Loaded {len(data)} questions: {n_mcq} MCQ, {n_frq} FRQ")
print(f"Data loading took {time.time() - t0:.2f}s")

# ── Prompts ────────────────────────────────────────────────────
# For this thinking-only model, do NOT try to suppress thinking.
# Instead, require standardized final output and parse after </think>.
SYSTEM_PROMPT_MCQ = (
    "You are solving a multiple-choice math problem. "
    "Reason efficiently and avoid unnecessary repetition. "
    "At the end, output exactly one final answer in the format \\boxed{A}. "
    "The boxed answer must contain only the choice letter."
)

SYSTEM_PROMPT_FRQ = (
    "You are solving a math problem. "
    "Reason efficiently and avoid unnecessary repetition. "
    "At the end, output exactly one final answer in the format \\boxed{...}. "
    "Do not add text after the boxed answer."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options)
        )
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"

    return SYSTEM_PROMPT_FRQ, question

def make_prompt(item: dict) -> str:
    system, user = build_prompt(item["question"], item.get("options"))

    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

def format_eta(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.0f}s"
    if seconds < 3600:
        return f"{seconds / 60:.1f}m"
    return f"{seconds / 3600:.1f}h"

print("Cell 1 complete.")

Cell 1: imports, config, data, prompts...
RUN_NAME                 = vllm_run2
RESPONSE_PATH            = results/vllm_run2_responses.jsonl
ATTEMPT_PATH             = results/vllm_run2_attempts.jsonl
EVAL_PATH                = results/vllm_run2_eval.jsonl
MAX_INPUT_LEN            = 8192
MAX_MODEL_LEN            = 8192
MCQ batch/toks           = 1, 16384
FRQ batch/toks           = 1, 16384
GPU_MEMORY_UTILIZATION   = 0.5
Loaded 16 questions: 7 MCQ, 9 FRQ
Data loading took 0.02s
Cell 1 complete.


In [3]:
print("Cell 2: loading tokenizer and vLLM model...")

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded in {time.time() - t0:.2f}s")

t0 = time.time()

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

print(f"Model loaded in {(time.time() - t0) / 60:.2f} min.")

Cell 2: loading tokenizer and vLLM model...
Tokenizer loaded in 1.23s
INFO 05-01 09:18:14 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.5, 'max_num_batched_tokens': 32768, 'max_num_seqs': 256, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-01 09:18:14 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 05-01 09:18:14 [model.py:1678] Using max model len 16384
INFO 05-01 09:18:14 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 05-01 09:18:32 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=24050) INFO 05-01 09:18:33 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=N

(EngineCore pid=24050) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=24050) <frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(EngineCore pid=24050) INFO 05-01 09:18:50 [bitsandbytes_loader.py:786] Loading weights with BitsAndBytes quantization. May take a while ...


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=24050) INFO 05-01 09:18:53 [gpu_model_runner.py:4820] Model loading took 2.7 GiB memory and 14.923680 seconds
(EngineCore pid=24050) INFO 05-01 09:19:14 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/8c9f128ca0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=24050) INFO 05-01 09:19:14 [backends.py:1111] Dynamo bytecode transform time: 20.25 s
(EngineCore pid=24050) INFO 05-01 09:19:17 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 32768) from the cache, took 2.523 s
(EngineCore pid=24050) INFO 05-01 09:19:17 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/8c300eb6da3e0ead728b359a7a6320e1b3efb48edaff76c86977facba6772916/rank_0_0/model
(EngineCore pid=24050) INFO 05-01 09:19:17 [monitor.py:48] torch.compile took 23.15 s in total
(EngineCore pid=24050) INFO 05-01 09:19:17 [monitor.py:76] Initial profiling/warmup run took 0.17 s
(Engin

(EngineCore pid=24050) 2026-05-01 09:19:20,239 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=24050) 2026-05-01 09:19:20,332 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 11.23it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 14.94it/s]


(EngineCore pid=24050) INFO 05-01 09:19:27 [gpu_model_runner.py:6046] Graph capturing finished in 8 secs, took 0.70 GiB
(EngineCore pid=24050) INFO 05-01 09:19:27 [gpu_worker.py:597] CUDA graph pool memory: 0.7 GiB (actual), 0.49 GiB (estimated), difference: 0.21 GiB (29.3%).
(EngineCore pid=24050) INFO 05-01 09:19:28 [core.py:283] init engine (profile, create kv cache, warmup model) took 34.47 seconds
Model loaded in 1.25 min.


In [4]:
print("Cell 3: generating responses with vLLM checkpointing...")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Using response file:", RESPONSE_PATH)

# Qwen official </think> token id
THINK_END_TOKEN_ID = 151668

def has_boxed(text: str) -> bool:
    return bool(re.search(r"\\boxed\{.*?\}", text, flags=re.DOTALL))

def extract_qwen_final_content_from_vllm(completion, raw_text: str):
    """
    vLLM gives both text and token_ids.
    For Qwen3 Thinking, reasoning comes before </think> token id 151668.
    If that token appears, keep only content after it.
    If it does not appear, fall back to raw text.
    """
    token_ids = getattr(completion, "token_ids", None)

    if token_ids is not None:
        ids = list(token_ids)

        try:
            idx = len(ids) - ids[::-1].index(THINK_END_TOKEN_ID)
            final_ids = ids[idx:]
            final_text = tokenizer.decode(final_ids, skip_special_tokens=True).strip()
            if final_text:
                return final_text, True
        except ValueError:
            pass

    # Fallback if token_ids not available or </think> missing
    if "</think>" in raw_text:
        return raw_text.split("</think>")[-1].strip(), True

    return raw_text.strip(), False

# ── Load existing complete responses from selected run file ─────
responses_by_id = {}

if RESPONSE_PATH.exists():
    with open(RESPONSE_PATH, "r") as f:
        for line in f:
            row = json.loads(line)
            responses_by_id[str(row["id"])] = row["response"]

print(f"Already completed: {len(responses_by_id)}/{len(data)}")

def save_responses_atomic():
    tmp_path = RESPONSE_PATH.with_suffix(".tmp")
    backup_path = RESPONSE_PATH.with_suffix(".bak")

    with open(tmp_path, "w") as f:
        for item in data:
            qid = str(item["id"])
            if qid in responses_by_id:
                f.write(json.dumps({
                    "id": item["id"],
                    "is_mcq": bool(item.get("options")),
                    "response": responses_by_id[qid],
                }) + "\n")

    if RESPONSE_PATH.exists():
        shutil.copy2(RESPONSE_PATH, backup_path)

    tmp_path.replace(RESPONSE_PATH)

def append_attempt(item, raw_text, final_text, reached_think_end, complete, max_tokens, label):
    with open(ATTEMPT_PATH, "a") as f:
        f.write(json.dumps({
            "id": item["id"],
            "is_mcq": bool(item.get("options")),
            "label": label,
            "max_tokens": max_tokens,
            "reached_think_end": reached_think_end,
            "complete": complete,
            "raw_preview": raw_text[:500],
            "final_text": final_text,
        }) + "\n")

def make_sampling_params(max_tokens: int) -> SamplingParams:
    return SamplingParams(
        max_tokens=max_tokens,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        min_p=0.0,
        presence_penalty=0.0,
        repetition_penalty=1.0,
    )

def generate_group(items, batch_size, max_tokens, label):
    remaining = [item for item in items if str(item["id"]) not in responses_by_id]

    print(f"\n{label}: {len(items) - len(remaining)}/{len(items)} already done")
    print(f"{label}: {len(remaining)} remaining")
    print(f"{label}: batch_size={batch_size}, max_tokens={max_tokens}")

    if not remaining:
        return

    sampling_params = make_sampling_params(max_tokens)

    group_start_time = time.time()
    pbar = tqdm(range(0, len(remaining), batch_size), desc=label)

    for batch_start in pbar:
        batch = remaining[batch_start:batch_start + batch_size]

        try:
            t_prompt = time.time()
            prompts = [make_prompt(item) for item in batch]
            prompt_time = time.time() - t_prompt

            t_gen = time.time()
            outputs = llm.generate(prompts, sampling_params=sampling_params)
            gen_time = time.time() - t_gen

            t_parse = time.time()

            saved_this_batch = 0
            incomplete_this_batch = 0

            for item, out in zip(batch, outputs):
                completion = out.outputs[0]
                raw_text = completion.text.strip()

                final_text, reached_think_end = extract_qwen_final_content_from_vllm(
                    completion=completion,
                    raw_text=raw_text,
                )

                complete = has_boxed(final_text)

                append_attempt(
                    item=item,
                    raw_text=raw_text,
                    final_text=final_text,
                    reached_think_end=reached_think_end,
                    complete=complete,
                    max_tokens=max_tokens,
                    label=label,
                )

                if complete or SAVE_INCOMPLETE:
                    responses_by_id[str(item["id"])] = final_text
                    saved_this_batch += 1
                else:
                    incomplete_this_batch += 1

            parse_time = time.time() - t_parse

            save_responses_atomic()

            completed_total = len(responses_by_id)
            completed_group = min(batch_start + len(batch), len(remaining))
            elapsed_group = time.time() - group_start_time

            avg_sec_per_remaining_item = elapsed_group / max(completed_group, 1)
            eta_group = avg_sec_per_remaining_item * (len(remaining) - completed_group)

            pbar.set_postfix({
                "saved": f"{completed_total}/{len(data)}",
                "batch_saved": saved_this_batch,
                "incomplete": incomplete_this_batch,
                "prompt_s": f"{prompt_time:.1f}",
                "gen_s": f"{gen_time:.1f}",
                "parse_s": f"{parse_time:.1f}",
                "ETA": format_eta(eta_group),
            })

        except Exception as e:
            print("\nGeneration crashed. Saving completed responses before raising error...")
            save_responses_atomic()
            print(f"Saved {len(responses_by_id)} complete responses to {RESPONSE_PATH}")
            raise e

        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

# ── Split MCQ and FRQ into separate groups ──────────────────────
mcq_items = [item for item in data if item.get("options")]
frq_items = [item for item in data if not item.get("options")]

total_start_time = time.time()

# MCQ first
generate_group(
    items=mcq_items,
    batch_size=BATCH_SIZE_MCQ,
    max_tokens=MAX_TOKENS_MCQ,
    label="MCQ",
)

# FRQ second
generate_group(
    items=frq_items,
    batch_size=BATCH_SIZE_FRQ,
    max_tokens=MAX_TOKENS_FRQ,
    label="FRQ",
)

# Build ordered lists for compatibility
responses = [responses_by_id[str(item["id"])] for item in data if str(item["id"]) in responses_by_id]
response_ids = [item["id"] for item in data if str(item["id"]) in responses_by_id]

print("\nGeneration cell complete.")
print(f"Generated/saved complete responses: {len(responses_by_id)}/{len(data)}")
print(f"Complete responses saved to: {RESPONSE_PATH}")
print(f"All attempts/debug saved to: {ATTEMPT_PATH}")
print(f"Total generation time this run: {format_eta(time.time() - total_start_time)}")

if len(responses_by_id) < len(data):
    print("Not all responses are complete yet.")
    print("Increase MAX_TOKENS_MCQ/MAX_TOKENS_FRQ or rerun Cell 3 to retry incomplete ones.")
else:
    print("All responses complete.")

Cell 3: generating responses with vLLM checkpointing...
Using response file: results/vllm_run2_responses.jsonl
Already completed: 0/16

MCQ: 0/7 already done
MCQ: 7 remaining
MCQ: batch_size=1, max_tokens=16384


MCQ:   0%|          | 0/7 [00:00<?, ?it/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Generation cell complete.
Generated/saved complete responses: 16/16
Complete responses saved to: results/vllm_run2_responses.jsonl
All attempts/debug saved to: results/vllm_run2_attempts.jsonl
Total generation time this run: 7.1m
All responses complete.


In [5]:
print("Cell 4: evaluation and score saving...")

t0 = time.time()

# ── Load responses from selected run file ───────────────────────
responses_by_id = {}

with open(RESPONSE_PATH, "r") as f:
    for line in f:
        row = json.loads(line)
        responses_by_id[str(row["id"])] = row["response"]

print(f"Loaded {len(responses_by_id)}/{len(data)} complete responses from {RESPONSE_PATH}")

missing = [item["id"] for item in data if str(item["id"]) not in responses_by_id]
if missing:
    print(f"Warning: {len(missing)} responses missing. First few missing ids: {missing[:10]}")

# ── Judger setup ───────────────────────────────────────────────
sys.path.insert(0, ".")
from judger import Judger

judger = Judger(strict_extract=False)

def extract_boxed(text: str) -> str:
    matches = re.findall(r"\\boxed\{([^{}]*)\}", text)
    return matches[-1].strip() if matches else ""

def extract_letter(text: str) -> str:
    boxed = extract_boxed(text)

    if re.fullmatch(r"[A-Za-z]", boxed):
        return boxed.upper()

    # JSON-ish fallback
    m = re.search(r'"answer"\s*:\s*"([A-Za-z])"', text)
    if m:
        return m.group(1).upper()

    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

def has_boxed_final(text: str) -> bool:
    return bool(re.search(r"\\boxed\{.*?\}", text, flags=re.DOTALL))

# ── Evaluate ───────────────────────────────────────────────────
results = []

eval_start = time.time()

for item in tqdm(data, desc="Scoring"):
    qid = str(item["id"])

    if qid not in responses_by_id:
        continue

    response = responses_by_id[qid]
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]

        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id": item["id"],
        "is_mcq": is_mcq,
        "gold": gold,
        "response": response,
        "has_boxed": has_boxed_final(response),
        "correct": correct,
    })

eval_time = time.time() - eval_start

# ── Save eval results ──────────────────────────────────────────
with open(EVAL_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

# ── Summary ────────────────────────────────────────────────────
mcq_res = [r for r in results if r["is_mcq"]]
frq_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

summary = {
    "run_name": RUN_NAME,
    "response_path": str(RESPONSE_PATH),
    "attempt_path": str(ATTEMPT_PATH),
    "eval_path": str(EVAL_PATH),
    "num_total_data": len(data),
    "num_evaluated": len(results),
    "num_missing": len(missing),
    "mcq_correct": sum(r["correct"] for r in mcq_res),
    "mcq_total": len(mcq_res),
    "mcq_accuracy": acc(mcq_res),
    "frq_correct": sum(r["correct"] for r in frq_res),
    "frq_total": len(frq_res),
    "frq_accuracy": acc(frq_res),
    "overall_correct": sum(r["correct"] for r in results),
    "overall_total": len(results),
    "overall_accuracy": acc(results),
    "num_with_boxed": sum(r["has_boxed"] for r in results),
    "num_without_boxed": sum(not r["has_boxed"] for r in results),
    "eval_time_sec": eval_time,
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"MCQ       : {summary['mcq_correct']:4d} / {summary['mcq_total']:4d} = {summary['mcq_accuracy']:.2f}%")
print(f"FRQ       : {summary['frq_correct']:4d} / {summary['frq_total']:4d} = {summary['frq_accuracy']:.2f}%")
print(f"Overall   : {summary['overall_correct']:4d} / {summary['overall_total']:4d} = {summary['overall_accuracy']:.2f}%")
print("-" * 50)
print(f"With boxed answer    : {summary['num_with_boxed']}")
print(f"Without boxed answer : {summary['num_without_boxed']}")
print(f"Missing responses    : {summary['num_missing']}")
print("-" * 50)
print(f"Saved eval to    : {EVAL_PATH}")
print(f"Saved summary to : {SUMMARY_PATH}")
print(f"Eval time        : {eval_time:.2f}s")
print("=" * 50)

print("Cell 4 complete.")

Cell 4: evaluation and score saving...
Loaded 16/16 complete responses from results/vllm_run2_responses.jsonl


Scoring:   0%|          | 0/16 [00:00<?, ?it/s]

EVALUATION RESULTS
MCQ       :    6 /    7 = 85.71%
FRQ       :    5 /    9 = 55.56%
Overall   :   11 /   16 = 68.75%
--------------------------------------------------
With boxed answer    : 16
Without boxed answer : 0
Missing responses    : 0
--------------------------------------------------
Saved eval to    : results/vllm_run2_eval.jsonl
Saved summary to : results/vllm_run2_summary.json
Eval time        : 0.80s
Cell 4 complete.
